# RoadGuard — Data Preparation

Prepare the ANOVOX Static Mono datasets listed in `configs/data_sources.yaml` for anomaly segmentation.

Each archive is **streamed from Zenodo** (no raw archive is stored on Drive): RGB, depth and semantic images are kept, anomaly masks are generated on the fly, and the result is saved as one package per archive in `RoadGuard_Processed/towns/`. The notebook then builds the master manifest and the leave-one-family-out (LOFO) splits.

To add a town, add one line to `configs/data_sources.yaml`. Already prepared archives are skipped on rerun.

All reusable processing logic resides in the roadguard Python package.

## 1. Connect Google Drive

Mount Google Drive to access the original ANOVOX archives and store the prepared dataset persistently.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Set up the RoadGuard environment

Clone the project repository and install its Python package in the Colab runtime. If installation requires a runtime restart, reconnect Drive and restore the working directory and imports as needed. Do not clone again if the repository is already present in the runtime.

In [ ]:
from google.colab import userdata
from pathlib import Path
import subprocess
import os
import base64

REPO = Path("/content/road-guard")

if not REPO.exists():
    token = userdata.get("GITHUB_TOKEN")

    credentials = base64.b64encode(
        f"x-access-token:{token}".encode()
    ).decode()

    env = os.environ.copy()
    env["GIT_CONFIG_COUNT"] = "1"
    env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {credentials}"

    subprocess.run(
        [
            "git", "clone",
            "https://github.com/RanaShoaaib/RoadGuard.git",
            str(REPO),
        ],
        check=True,
        env=env,
    )

    del token, credentials, env

os.chdir(REPO)
print("Repository:", REPO)

## 2b. Install the package

Install the repository in editable mode and verify the import.

In [ ]:
%pip install -e .

# Make the RoadGuard package importable from its src directory.
# Temporary workaround for the Colab editable-install issue.
import sys
sys.path.insert(0, str(REPO / "src"))

# Verify the import before continuing.
from roadguard.data.preparation import prepare_archive
print("RoadGuard imported successfully")

## 3. Configure data locations

`configs/data_sources.yaml` lists the archives (name and Zenodo record) and the split settings. `RoadGuard_Processed` is the persistent Drive folder for the prepared data; `STAGE_DIR` is a temporary folder on the Colab disk where one archive at a time is unpacked, checked and packaged.

In [ ]:
from pathlib import Path
import pandas as pd

from roadguard.data.preparation import (
    load_data_config,
    prepare_archive,
    load_archive_manifests,
    stage_prepared_dataset,
)
from roadguard.data.splits import create_lofo_splits, summarize_folds

CONFIG_PATH = REPO / "configs" / "data_sources.yaml"

# Persistent folder for prepared data and experiment splits
OUTPUT_DIR = Path("/content/drive/MyDrive/RoadGuard_Processed")
PROCESSED_DIR = OUTPUT_DIR / "processed"

# Temporary working folder on the Colab disk
STAGE_DIR = Path("/content/rg_stage")

config = load_data_config(CONFIG_PATH)
print("Archives:", [a.name for a in config["archives"]])
print("Split settings:", config["splits"])

## 4. Prepare the archives

For every configured archive: stream it from Zenodo, write the RGB, depth and semantic images and the generated anomaly masks, build and validate its manifest, and package it as `towns/<name>.tar` with a manifest CSV and a JSON sidecar. The checksum published by Zenodo is verified.

Each archive takes about 1 to 1.5 hours (the download is the bottleneck). Keep the browser tab open. If the runtime disconnects, rerun this cell: finished archives are skipped, and an archive whose stream completed but whose packaging failed continues from the local stage. Scenarios in which the anomaly is never visible are dropped and listed in the sidecar.

In [ ]:
results = []

for spec in config["archives"]:
    result = prepare_archive(
        spec,
        output_dir=OUTPUT_DIR,
        stage_dir=STAGE_DIR,
        anomaly_ids=config["anomaly_ids"],
    )
    results.append(result)
    print(f"{spec.name}: {result['status']} — {result['scenarios']} scenarios, "
          f"{result['frames']} frames, {result['tar_gb']} GB")

## 5. Review the prepared archives

Check scenario and frame counts, the anomaly families found in each archive, the dropped scenarios, and that the download checksum was verified.

In [ ]:
summary = pd.DataFrame([
    {
        "archive": r["name"],
        "status": r["status"],
        "scenarios": r["scenarios"],
        "frames": r["frames"],
        "families": r["families"],
        "dropped_empty": len(r["dropped_scenarios_without_anomaly"]),
        "md5_checked": r["md5_checked"],
        "tar_gb": r["tar_gb"],
    }
    for r in results
])
display(summary)

## 6. Build the master manifest

Combine the per-archive manifests into one frame-level table. Image paths are stored relative to the extracted dataset root.

In [ ]:
names = [a.name for a in config["archives"]]
manifest = load_archive_manifests(OUTPUT_DIR, names)

print(f"Manifest frames: {len(manifest)}")
display(
    manifest.drop_duplicates("scenario_id")
    .groupby("anomaly_family")
    .agg(scenarios=("scenario_id", "size"), objects=("anomaly_object", "nunique"))
)
print("Scenarios per town:")
print(manifest.drop_duplicates("scenario_id").groupby("town").size())
print(f"Frames without anomaly pixels: {(~manifest['has_anomaly']).mean():.1%}")

## 7. Save the master manifest

Persist the validated frame index for reuse in training and evaluation notebooks.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = PROCESSED_DIR / "master_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print(f"Saved {len(manifest)} frames to {manifest_path}")

## 8. Create reproducible dataset splits

Create leave-one-family-out (LOFO) splits. Each fold holds out one anomaly family for testing. Validation uses whole anomaly objects of the other families, so an object is never in training and validation together. Families with fewer than `min_family_scenarios` scenarios are always training data. Frames without any visible anomaly stay with their scenario, so every split contains negative frames.

In [ ]:
folds = create_lofo_splits(manifest, **config["splits"])

print(f"Generated {len(folds)} LOFO folds")
print("Held-out anomaly families:", list(folds))
display(summarize_folds(folds))

## 9. Save split assignments

Save one complete frame-level CSV per LOFO fold and a fold summary. These files define the exact partitions used by downstream experiments.

In [ ]:
SPLITS_DIR = PROCESSED_DIR / "lofo_splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

for family, fold in folds.items():
    fold.to_csv(SPLITS_DIR / f"{family}.csv", index=False)

summarize_folds(folds).to_csv(PROCESSED_DIR / "fold_summary.csv", index=False)

print(f"Saved {len(folds)} LOFO folds to {SPLITS_DIR}")

## 10. Verify the dataset can be staged for training

Unpack the packages onto the Colab disk exactly as the training notebook does, and confirm that every file in the manifest exists. This takes some minutes and about the size of the packages in local disk space; set `VERIFY_STAGING = False` to skip it. The training notebook stages the data itself.

In [ ]:
VERIFY_STAGING = True

if VERIFY_STAGING:
    staged = stage_prepared_dataset(OUTPUT_DIR, "/content/roadguard_data")
    missing = [
        p for column in ("rgb_path", "depth_path", "semantic_path", "mask_path")
        for p in manifest[column]
        if not (staged["data_root"] / p).is_file()
    ]
    assert not missing, f"{len(missing)} manifest files missing, e.g. {missing[:3]}"
    print("All manifest files are present in the staged dataset:", staged["data_root"])
    print("Fold files:", sorted(p.name for p in (staged["processed_dir"] / "lofo_splits").glob("*.csv")))